In [66]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import sys
import json
from datetime import datetime
import numpy as np


In [67]:
from typing import get_origin, get_args, Literal

In [68]:
MODEL = "qwen3:4b"

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)


In [69]:
class Memory:
    def __init__(self, embedding_function, file_path = "memory.json"):
        self.data = []
        self.next_id = 1
        self.embedding_function = embedding_function
        self.file_path = file_path

        self.load_from_disk()

    def save_to_disk(self):
        with open(self.file_path, "w", encoding="utf-8") as f:
            json.dump(self.data, f, indent=4)

    def load_from_disk(self):
        if not os.path.exists(self.file_path):
            return f"File path does not exist!!"

        with open(self.file_path, "r", encoding = "utf-8") as f:
            self.data = json.load(f)

        if self.data:
            self.next_id = max(memory["id"] for memory in self.data) + 1
            

    def remember(self, key, value):
        text = f"{key} : {value}"
        embedding = self.embedding_function(text)

        for memory in self.data:

            if memory["key"] == key:
                memory["value"] = value
                memory["text"] = text
                memory["embedding"] = embedding
                memory["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                self.save_to_disk()

                return

        self.data.append({
            "id" : self.next_id,
            "key": key,
            "value": value,
            "text" : f"{key} : {value}",
            "embedding": embedding,
            "timestamp" : datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })
        
        self.next_id += 1
        self.save_to_disk()
    
    def recall(self, key):
        for memory in reversed(self.data):
            if memory["key"] == key:
                return memory["value"]

    def forget(self, key):
        self.data = [
            memory
            for memory in self.data
            if memory["key"] != key
        ]

    def list_memories(self):
        return self.data

    def search(self, query, top_k = 3, threshold = 0.5):
        query_embedding = self.embedding_function(query)

        results = []

        for memory in self.data:
            score = cosine_similarity(query_embedding, memory["embedding"])

            if score >= threshold:
                results.append({
                    "memory" : memory,
                    "score" : score
                })

        results.sort(
            key = lambda x : x["score"],
            reverse = True
        )

        return results[:top_k]


In [70]:
class Tool:
    def __init__(self, function, description):
        self.function = function
        self.description = description
        self.parameters = generate_parameters(function)

    def execute(self, arguments, context):
        try:
            if context is None:
                context = {}
            
            return self.function(**arguments, **context)
        except Exception as e:
            return f"Tool execution failed: {str(e)}"

    def schema(self):
        return {
            "type" : "function",
            "function":{
                "name": self.function.__name__,
                "description": self.description,
                "parameters": self.parameters
            }
        }
    

In [71]:
class Agent:
    def __init__(self, client, tool_list, model, system_prompt):
        self.client = client
        self.model = model
        self.tool_list = tool_list

        self.TOOL_MAP = {
            tool.function.__name__ : tool
            for tool in tool_list
        }

        self.Tool_SCHEMA = [
            tool.schema()
            for tool in tool_list
        ]

        self.messages = [{
            "role": "system",
            "content": system_prompt
        }]

        self.state = {}
        self.memory = Memory(create_embedding, file_path = "memory.json")
        self.context = {"memory": self.memory}
        
    def set_state(self, key, value):
        self.state[key] = value

    def get_state(self, key):
        return self.state[key]

    def call_llm(self):
            return self.client.chat.completions.create(
                    model=self.model,
                    messages=self.messages,
                    tools=self.Tool_SCHEMA,
                    max_tokens=1000
                )

    def execute(self, tool_call):
            tool_name = tool_call.function.name
            print(f"TOOL CALLED: {tool_name}")
            tool = self.TOOL_MAP.get(tool_name)
            if not tool:
                return f"Tool '{tool_name}' does not exist."
            try:
                arguments = json.loads(tool_call.function.arguments)
                return tool.execute(arguments, self.context)
            except Exception as e:
                return f"Tool execution failed: {str(e)}"
        
    def run(self, user_input):
        self.messages.append({"role": "user", "content": user_input})
        MAX_ITERATIONS = 10
        for iteration in range(MAX_ITERATIONS):
            response = self.call_llm()
            response_message = response.choices[0].message
            self.messages.append(response_message)
            if not response_message.tool_calls:
                return f"AI: ",response_message.content
            for tool_call in response_message.tool_calls:
                    print("Tool is running")
                    result = self.execute(tool_call)
                    self.messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": tool_call.function.name,
                        "content": json.dumps(result)
                    })
        else:
            print("MAX Tool iterations reached!!")


In [72]:
EMBEDDING_MODEL = "Qwen3-Embedding:4B"

embedding_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)


In [73]:
def create_embedding(text):
    embedding = embedding_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text,
        encoding_format="float"
    )
    return embedding.data[0].embedding


In [74]:
def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)

    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [75]:
def python_type_to_json_type(annotation):

    if get_origin(annotation) is Literal:

        values = get_args(annotation)

        first_value = values[0]

        if isinstance(first_value, str):
            json_type = "string"
        
        elif isinstance(first_value, int):
            json_type = "integer"

        elif isinstance(first_value, float):
            json_type = "number"
        
        elif isinstance(first_value, bool):
            json_type = "boolean"

        else:
            json_type = "string"

        return{
            "type": json_type,
            "enum": list(values)
        }
    if annotation == str:
        return "string"

    elif annotation == int:
        return "integer"

    elif annotation == float:
        return "number"

    elif annotation == bool:
        return "boolean"

    return "string"

In [76]:
import inspect

def generate_parameters(function):
    
    signature = inspect.signature(function)

    properties = {}
    required = []

    for name, parameter in signature.parameters.items():
        if name == "memory":
            continue

        json_type = python_type_to_json_type(parameter.annotation)

        properties[name] = {
            "type" : json_type
        }

        if parameter.default is inspect.Parameter.empty:
            required.append(name)

    return {
        "type" : "object",
        "properties" : properties,
        "required" : required
    }

In [77]:
def get_current_time():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [78]:
time_tool = Tool(
    function = get_current_time,
    description="Get current Time",
)

In [79]:
def calculator(a: float, b: float, operation: Literal["add", "subtract", "multiply", "divide"]):
    if(operation == "add"):
        return a+b

    elif(operation == "divide"):
        if(b != 0):
            return a/b
        else: return "Cannot divide with Zero"

    elif(operation == "subtract"):
        return a-b
    
    elif(operation == "multiply"):
        return a*b


In [80]:
calculator_tool = Tool(
    function=calculator,
    description="Perform mathematical calculations",
)

In [81]:
def greet(name: str, age: int, excited: bool = False):
    if excited:
        return f"Hello {name}! You are {age} years old!"
    return f"Hello {name}. You are {age} years old."

In [82]:
greet_tool = Tool(
    greet,
    "Greets a Person"
)

In [83]:
def save_memory(memory: Memory, key: str, value: str):
    memory.remember(key, value)
    return f"Remembered {key} = {value}"

In [84]:
memory_tool = Tool(
    save_memory,
    "Saves important information to agent's memory"
)

In [85]:
def recall_memory(memory: Memory, key: str):
    value = memory.recall(key)
    if value is None:
        return f"No memory found for '{key}'"

    return f"The value of {key} = {value}"

In [86]:
recall_tool = Tool(
    recall_memory,
    """Retrieve a memory using its exact key.

    Use this ONLY when you already know the exact memory key.
    For example, if the key is exactly "hometown", use:
    recall_memory(key="hometown").

    If you do not know the exact key, use search_memory instead."""
)

In [87]:
def forget_memory(memory: Memory, key: str):
    memory.forget(key)
    return f"memory forgotten"

In [88]:
forget_tool = Tool(
    forget_memory,
    "Used to forget a memory from agent's memory"
)

In [89]:
def get_all_memories(memory: Memory):
    return memory.list_memories()

In [90]:
def search_memory(memory: Memory, query: str):
    print("Using search memory")
    results =  memory.search(query)

    if not results:
        return {
            "found": False,
            "memories": []
            }

    cleaned_results = []

    for result in results:
        memory_data = result["memory"]

        cleaned_results.append({
            "key": memory_data["key"],
            "value": memory_data["value"],
            "score": round(result["score"], 3)
        })

    return {
        "found": True,
        "memories": cleaned_results
    }

In [91]:
search_memory_tool = Tool(
    search_memory,
    """Search the agent's memory using keywords when you are unsure of the
    exact memory key. Use this tool when the user asks about something
    that may be stored in memory but you do not know the exact key.

    Example:
    User asks "What programming language do I like?"
    Search using query="language".

    Do NOT use recall_memory unless you know the exact key."""
)

In [92]:
list_memory_tool = Tool(
    get_all_memories,
    "get all the memories currently stored by the agent"
)

In [93]:
tool_list = [
    calculator_tool,
    time_tool,
    greet_tool,
    memory_tool,
    recall_tool,
    forget_tool,
    search_memory_tool
]

In [94]:
agent = Agent(
    client=client,
    tool_list=tool_list,
    model=MODEL,
    system_prompt="""You are a helpful AI agent.

You have access to tools for calculations, getting the current time,
and managing memory.

Use the calculator for mathematical calculations.
Use the time tool when the user asks for the current time.

Memory rules:

1. When the user explicitly asks you to remember something,
   use save_memory.

3. If you do NOT know the exact memory key, use search_memory.
   Do not guess the key.

4. Never use get_all_memories.

5. Do not invent memories."""
)


In [95]:
while True:
    try:
        user_input = input("You: ")
    except (EOFError, KeyboardInterrupt):
        break

    if not user_input.strip():
        continue

    if user_input.lower().strip() == "exit":
        break

    try:
        response = agent.run(user_input)
        print(response)
    except Exception as e:
        print("Error:", e)


In [96]:
agent.memory.remember("favorite_language", "Python")
agent.memory.remember("hometown", "Vizag")
agent.memory.remember("college", "ABC College")

APIConnectionError: Connection error.

In [ ]:
results = agent.memory.search(
    "where do i live?"
)

print(results)

[{'memory': {'id': 2, 'key': 'hometown', 'value': 'Vizag', 'text': 'hometown : Vizag', 'embedding': [-0.0003734558, -0.001335764, -0.0077888737, 0.0047036805, -0.0015297055, 0.058387876, 0.06701341, 0.015186501, 0.012589619, 0.043306775, 0.018231114, 0.014931634, -4.281477e-05, -0.04399922, 0.060821105, 0.037726145, 0.025788436, -0.07868276, -0.023367489, -0.0018783197, -0.010580301, 0.009587014, 0.03777014, 0.021446671, -0.015469372, 0.012551355, -0.023357082, -0.0458027, 0.013327388, -0.009847971, -0.055424374, -0.016777424, 0.06542019, -0.0055130282, 0.0022913988, -0.00046271284, -0.01269419, -0.010214442, -0.0064019924, -0.005510832, 0.03458784, -0.02244912, 0.016126914, -0.0077411826, 0.025498817, 0.0010898599, -0.0060931803, 0.0053505837, -0.007566791, -0.01884718, 0.016051758, -0.0036038805, -0.026449796, -0.019759193, 0.018808022, 0.03698931, 0.028030062, -0.0115833, -0.026190622, 0.0008571212, 0.0014521743, 0.017771687, -0.0067010294, -0.03309075, -0.003585215, -0.029364463, 0

In [ ]:
results = agent.memory.search(
    "What programming language do I like?"
)

for result in results:
    print(
        result["memory"]["key"],
        "â†’",
        result["memory"]["value"],
        "|",
        result["score"]
    )

favorite_language â†’ Python | 0.8050385100392544


In [ ]:
results = agent.memory.search(
    "What is my favorite food?"
)

print(results)

[{'memory': {'id': 1, 'key': 'favorite_language', 'value': 'Python', 'text': 'favorite_language : Python', 'embedding': [-8.8736924e-05, 0.0017507928, 0.015706731, 0.011106292, -0.00048386655, 0.020070534, 0.110649414, 0.002484709, 0.025754586, -1.4662572e-05, 0.071155176, -0.004437144, 0.0003062169, 0.0013436293, 0.01453251, -0.0010962031, 0.004401249, -0.034014925, -0.016752167, -0.008348847, -0.015823783, -0.03713227, 0.0582778, 0.037622888, -0.008482596, 0.019563323, -0.020856993, -0.07116089, 0.020861791, 0.024951244, 0.019182406, -0.02772112, 0.026991533, 0.002658759, 0.0025496078, -0.02441114, -0.015027883, -0.0068335533, -0.003904281, -0.030958699, 0.003239299, -0.021115212, 0.029196566, 0.022741258, 0.012017829, -0.026349287, 0.0018994309, -0.023856482, 0.0043789204, -0.030350013, 0.0009743579, 0.0025395616, 0.0095073525, -0.029730732, 0.006584657, 0.020657694, 0.020784156, -0.011948934, -0.037912052, -0.011312478, -0.0037727873, -0.001632461, -0.0214663, -0.015886828, -0.0039

In [ ]:
agent.memory.remember(
    "favorite_language",
    "Python"
)

In [ ]:
queries = [
    "What programming language do I like?",
    "What is my favorite food?",
    "Where do I live?",
    "What is my name?",
    "What is my favorite color?"
]

for query in queries:
    results = agent.memory.search(
        query,
        top_k=1,
        threshold=0
    )

    print(query)

    for result in results:
        print(
            "   ",
            result["memory"]["key"],
            "→",
            result["memory"]["value"],
            "| score:",
            result["score"]
        )

    print()


What programming language do I like?
    favorite_language → Python | score: 0.8050385100392544



What is my favorite food?
    favorite_language → Python | score: 0.5109993940618403

Where do I live?
    hometown → Vizag | score: 0.5212000515182617

What is my name?
    favorite_language → Python | score: 0.5387502133302594

What is my favorite color?
    favorite_language → Python | score: 0.5349974944018917



In [ ]:
print(agent.memory.list_memories())

[{'id': 1, 'key': 'favorite_language', 'value': 'Python', 'text': 'favorite_language : Python', 'embedding': [-8.8736924e-05, 0.0017507928, 0.015706731, 0.011106292, -0.00048386655, 0.020070534, 0.110649414, 0.002484709, 0.025754586, -1.4662572e-05, 0.071155176, -0.004437144, 0.0003062169, 0.0013436293, 0.01453251, -0.0010962031, 0.004401249, -0.034014925, -0.016752167, -0.008348847, -0.015823783, -0.03713227, 0.0582778, 0.037622888, -0.008482596, 0.019563323, -0.020856993, -0.07116089, 0.020861791, 0.024951244, 0.019182406, -0.02772112, 0.026991533, 0.002658759, 0.0025496078, -0.02441114, -0.015027883, -0.0068335533, -0.003904281, -0.030958699, 0.003239299, -0.021115212, 0.029196566, 0.022741258, 0.012017829, -0.026349287, 0.0018994309, -0.023856482, 0.0043789204, -0.030350013, 0.0009743579, 0.0025395616, 0.0095073525, -0.029730732, 0.006584657, 0.020657694, 0.020784156, -0.011948934, -0.037912052, -0.011312478, -0.0037727873, -0.001632461, -0.0214663, -0.015886828, -0.003979674, -0.0

In [ ]:
test_memory = Memory(create_embedding)
print(test_memory.list_memories())

[{'id': 1, 'key': 'favorite_language', 'value': 'Python', 'text': 'favorite_language : Python', 'embedding': [-8.8736924e-05, 0.0017507928, 0.015706731, 0.011106292, -0.00048386655, 0.020070534, 0.110649414, 0.002484709, 0.025754586, -1.4662572e-05, 0.071155176, -0.004437144, 0.0003062169, 0.0013436293, 0.01453251, -0.0010962031, 0.004401249, -0.034014925, -0.016752167, -0.008348847, -0.015823783, -0.03713227, 0.0582778, 0.037622888, -0.008482596, 0.019563323, -0.020856993, -0.07116089, 0.020861791, 0.024951244, 0.019182406, -0.02772112, 0.026991533, 0.002658759, 0.0025496078, -0.02441114, -0.015027883, -0.0068335533, -0.003904281, -0.030958699, 0.003239299, -0.021115212, 0.029196566, 0.022741258, 0.012017829, -0.026349287, 0.0018994309, -0.023856482, 0.0043789204, -0.030350013, 0.0009743579, 0.0025395616, 0.0095073525, -0.029730732, 0.006584657, 0.020657694, 0.020784156, -0.011948934, -0.037912052, -0.011312478, -0.0037727873, -0.001632461, -0.0214663, -0.015886828, -0.003979674, -0.0

In [97]:
agent.memory.list_memories()

[{'id': 1,
  'key': 'favorite_language',
  'value': 'Python',
  'text': 'favorite_language : Python',
  'embedding': [-8.8736924e-05,
   0.0017507928,
   0.015706731,
   0.011106292,
   -0.00048386655,
   0.020070534,
   0.110649414,
   0.002484709,
   0.025754586,
   -1.4662572e-05,
   0.071155176,
   -0.004437144,
   0.0003062169,
   0.0013436293,
   0.01453251,
   -0.0010962031,
   0.004401249,
   -0.034014925,
   -0.016752167,
   -0.008348847,
   -0.015823783,
   -0.03713227,
   0.0582778,
   0.037622888,
   -0.008482596,
   0.019563323,
   -0.020856993,
   -0.07116089,
   0.020861791,
   0.024951244,
   0.019182406,
   -0.02772112,
   0.026991533,
   0.002658759,
   0.0025496078,
   -0.02441114,
   -0.015027883,
   -0.0068335533,
   -0.003904281,
   -0.030958699,
   0.003239299,
   -0.021115212,
   0.029196566,
   0.022741258,
   0.012017829,
   -0.026349287,
   0.0018994309,
   -0.023856482,
   0.0043789204,
   -0.030350013,
   0.0009743579,
   0.0025395616,
   0.0095073525,
   -